# Redraw the figures of the subphenotype analysis

**What it does.** After one complete run of `AE_subphenotypes_analysis.ipynb` in this folder, this notebook redraws every figure
(`output_subphenotypes/figures/`, with `figure_legends.md`) from the results that run saved in `cache_figure_state/`, in about a
minute. It runs the figure cells of `AE_subphenotypes_analysis.ipynb` as they are in that file now, so a change to a figure cell, or
a new version of the notebook that changes only figures, can be seen without rerunning the analysis. Nothing is recomputed: every
number comes from the saved run. If the analysis cells of the notebook differ from those of the saved run, it stops and asks for a
full run.

**How to run.** Keep this notebook in the same folder as `AE_subphenotypes_analysis.ipynb` and `cache_figure_state/`, close any
figure open in a PDF viewer, and run all cells. If the folder holds more than one copy of the analysis notebook (for example
`AE_subphenotypes_analysis (1).ipynb` from a second download), the copy with the newest figure code is used; the first output line
names the file and its figure code version. Close the analysis notebook in Jupyter before replacing it, so that an open tab
cannot save the old version over the new one.

In [ ]:
import ast, json, re, sys, time, pickle, hashlib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
HERE = Path.cwd()
STATE_MARKER = '# Saved results for redrawing the figures.'           # first line of the cell that saves the results
END_MARKER = '# Run log, then the identifier check over every output file.'   # first line of the cell after the figures
STATE_FILE = HERE / 'cache_figure_state' / 'figure_state_INTERNAL.pkl'
if not STATE_FILE.exists():
    raise FileNotFoundError('cache_figure_state/figure_state_INTERNAL.pkl not found: run AE_subphenotypes_analysis.ipynb in full once first.')
def _figure_code_version(p):
    """FIGURE_CODE_VERSION of an analysis notebook (empty for versions before 2026-10-15)."""
    _m = re.search(r"FIGURE_CODE_VERSION = '([0-9-]+)'", p.read_text(encoding='utf-8', errors='ignore'))
    return _m.group(1) if _m else ''


# Every copy of the analysis notebook in this folder (a second download may be named "AE_subphenotypes_analysis (1).ipynb"):
# the newest figure code wins, then the most recently modified file.
_nbs = sorted(HERE.glob('AE_subphenotypes_analysis*.ipynb'), key=lambda p: (_figure_code_version(p), p.stat().st_mtime), reverse=True)
if not _nbs:
    raise FileNotFoundError('AE_subphenotypes_analysis.ipynb must be in this folder.')
NB_FILE = _nbs[0]
FIG_CODE = _figure_code_version(NB_FILE) or 'before 2026-10-15'
CELLS = [''.join(c['source']) if isinstance(c['source'], list) else c['source']
         for c in json.loads(NB_FILE.read_text(encoding='utf-8'))['cells'] if c['cell_type'] == 'code']
I_STATE = next(i for i, s in enumerate(CELLS) if s.lstrip().startswith(STATE_MARKER))
I_END = next(i for i, s in enumerate(CELLS) if s.lstrip().startswith(END_MARKER))
with open(STATE_FILE, 'rb') as _fh:
    SAVED = pickle.load(_fh)
_digest = hashlib.sha256('\n'.join(ast.dump(ast.parse(s)) for s in CELLS[:I_STATE]).encode('utf-8')).hexdigest()[:16]
if SAVED['analysis_code'] is not None and _digest != SAVED['analysis_code']:
    raise RuntimeError(f"The analysis cells of {NB_FILE.name} differ from those of the saved run ({SAVED['saved']}). "
                       'Run AE_subphenotypes_analysis.ipynb in full again; the redraw is only for changes to figure cells.')
print(f"Saved run: {SAVED['saved']}. Figure cells from {NB_FILE.name} (figure code version {FIG_CODE}; modified "
      f"{time.strftime('%Y-%m-%d %H:%M', time.localtime(NB_FILE.stat().st_mtime))}): {I_END - I_STATE - 1}.")
if len(_nbs) > 1:
    print('Other copies of the analysis notebook in this folder, not used:',
          ', '.join(f'{p.name} (figure code version {_figure_code_version(p) or "before 2026-10-15"})' for p in _nbs[1:]))

In [ ]:
# The saved results, the paths of this folder, then the imports and function definitions of the analysis cells (definitions only:
# no analysis step is run again).
globals().update(SAVED['objects'])
FREEZE = HERE / 'AE_freeze_v1.4'
if not (FREEZE / 'manifest.json').exists() and (FREEZE / 'AE_freeze_v1.4' / 'manifest.json').exists():
    FREEZE = FREEZE / 'AE_freeze_v1.4'
OUT = HERE / 'output_subphenotypes'
TAB, FIG, LOGD, CODE = OUT / 'tables', OUT / 'figures', OUT / 'logs', OUT / 'code'
for _p in (OUT, TAB, FIG, LOGD, CODE):
    _p.mkdir(parents=True, exist_ok=True)
PROGRESS = LOGD / 'redraw_figures.log'
_tools = CODE / 'ae_lpa_tools.py'
if not _tools.exists() or _tools.read_text(encoding='utf-8') != TOOLS_SOURCE:
    _tools.write_text(TOOLS_SOURCE, encoding='utf-8')               # the module of the saved run
sys.dont_write_bytecode = True
for _p in (str(FREEZE / 'code'), str(CODE)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
for _i, _src in enumerate(CELLS[:I_STATE]):
    for _node in ast.parse(_src).body:
        if isinstance(_node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef)):
            try:
                exec(compile(ast.Module(body=[_node], type_ignores=[]), f'analysis cell {_i + 1}', 'exec'), globals())
            except ImportError as _e:
                print(f'Note: {_e} (not needed for the figures)')
_now = {'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': __import__('matplotlib').__version__}
_diff = {k: (SAVED['software'].get(k), v) for k, v in _now.items() if SAVED['software'].get(k) != v}
if _diff:
    print('Note: package versions differ from the saved run (saved, now):', _diff)
with open(PROGRESS, 'a', encoding='utf-8') as _fh:
    _fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} Redraw from the run saved {SAVED['saved']}, figure cells of {NB_FILE.name} (figure code version {FIG_CODE})\n")

In [ ]:
# The figure cells of the analysis notebook, run on the saved results.
for _i, _src in enumerate(CELLS[I_STATE + 1:I_END]):
    exec(compile(_src, f'figure cell {_i + 1}', 'exec'), globals())

In [ ]:
# Identifier check over the rewritten text files, as at the end of the analysis notebook.
_ids = set(D.index.astype(str))
_dates = set(D.admit_time.dt.strftime('%Y-%m-%d'))
_hits = []
for _f in list(FIG.glob('*.md')) + [PROGRESS]:
    _txt = _f.read_text(encoding='utf-8-sig')
    if [t for t in set(re.findall(r'\b\d{6,}\b', _txt)) if t in _ids] or any(d in _txt for d in _dates):
        _hits.append(_f.name)
if _hits:
    raise RuntimeError(f'Research numbers or admission dates found in: {_hits}')
log('Redraw', 'Identifier check', 'no research numbers or admission dates in the figure legends or this log')
print(f'Done: figures redrawn in {FIG.relative_to(HERE)}.')